# 7. Results and Cross-Method Comparison

This section consolidates every run that was launched on the SeqTO-v2 C-core domain and pits them against each other and against the random-search ceiling (Section 6.8). Seven RL runs + one random baseline + one random-policy reference are in scope:

* `dqn_run` — custom DQN, completed 200/200
* `a2c_run` — custom A2C with raw rewards (collapsed — negative control)
* `a2c_v2_run` — custom A2C with reward_scale=0.001 (stalled at ep 50 best)
* `ppo_run` — custom PPO with raw rewards (collapsed — negative control)
* `ppo_v2_run` — custom PPO with reward_scale=0.001 (productive, best ep 143)
* `sb3_a2c_run` — Stable-Baselines3 A2C reference (stuck at ep 2)
* `sb3_ppo_run` — Stable-Baselines3 PPO reference (productive but undertrained)
* `random_search` — 200 random masks across 4 density levels (Section 6.8)


## 7.1 All-methods overlay

Single figure: running-best end-of-episode force for every RL run, with the 200-sample random-search ceiling drawn as a horizontal reference line.


In [ ]:
# Cross-method overlay of running-max force vs episode
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

runs = [
    ('DQN',           'designs/dqn_run',     '#1f77b4', '-',  2.5),
    ('PPO-v2 (scaled)','designs/ppo_v2_run', '#2ca02c', '-',  2.0),
    ('A2C-v2 (scaled)','designs/a2c_v2_run', '#9467bd', '-',  2.0),
    ('SB3-PPO',       'designs/sb3_ppo_run', '#17becf', '-.', 1.5),
    ('SB3-A2C',       'designs/sb3_a2c_run', '#bcbd22', '-.', 1.5),
    ('PPO (raw)',     'designs/ppo_run',     '#d62728', '--', 1.2),
    ('A2C (raw)',     'designs/a2c_run',     '#ff7f0e', '--', 1.2),
]

rs_files = sorted(Path('designs/env_probe').glob('random_search_*.json'))
rs_max = json.loads(rs_files[-1].read_text())['summary']['max'] if rs_files else None
rs_mean = json.loads(rs_files[-1].read_text())['summary']['mean'] if rs_files else None

fig, ax = plt.subplots(figsize=(13, 7))
for label, d, color, ls, lw in runs:
    s = json.loads(open(f'{d}/training_stats.json').read())
    eps = s['episode_stats']
    xs = np.arange(1, len(eps) + 1)
    f  = np.array([e['force_N'] for e in eps])
    running_max = np.maximum.accumulate(f)
    ax.plot(xs, running_max, color=color, ls=ls, lw=lw,
            label=f'{label}  (n={len(eps)}, max={f.max():.0f} N)')

if rs_max is not None:
    ax.axhline(rs_max, color='black', ls=':', lw=1.5,
               label=f'random-search max ({rs_max:.0f} N)')
    ax.axhline(rs_mean, color='gray', ls=':', lw=1.0,
               label=f'random-search mean ({rs_mean:.0f} N)')

ax.set_xlabel('Episode'); ax.set_ylabel('Running-max end-of-episode force (N)')
ax.set_title('SeqTO-v2 C-core: running-best force per method  (all runs on same env, force-reward)')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7.2 Scorecard

Quantitative summary of every run, sorted by peak force.


In [ ]:
# Build the final scorecard from the training_stats.json files
import json
from pathlib import Path
import numpy as np

rs_files = sorted(Path('designs/env_probe').glob('random_search_*.json'))
rs_max = json.loads(rs_files[-1].read_text())['summary']['max'] if rs_files else None

RUNS = [
    ('DQN',            'designs/dqn_run',     'completed'),
    ('PPO-v2',         'designs/ppo_v2_run',  'still climbing'),
    ('A2C-v2',         'designs/a2c_v2_run',  'plateau confirmed'),
    ('SB3-PPO',        'designs/sb3_ppo_run', 'undertrained'),
    ('SB3-A2C',        'designs/sb3_a2c_run', 'stuck @ ep 2'),
    ('PPO (raw)',      'designs/ppo_run',     'collapsed'),
    ('A2C (raw)',      'designs/a2c_run',     'collapsed'),
]

rows = []
for label, d, status in RUNS:
    s = json.loads(open(f'{d}/training_stats.json').read())
    eps = s['episode_stats']; hp = s['hyperparams']
    f = np.array([e['force_N'] for e in eps])
    peak = f.max(); peak_ep = int(np.argmax(f)) + 1
    rows.append((label, len(eps), hp.get('episodes', '-'), peak, peak_ep, peak/rs_max if rs_max else float('nan'), status))
rows.sort(key=lambda r: r[3], reverse=True)

# Random-search baseline as a row
if rs_max is not None:
    rows.append(('Random-search',  200, '-', rs_max, '-', 1.0, 'baseline'))

print(f"{'Method':<14s} {'eps':>6s} {'plan':>6s} {'peak F':>8s} {'@ep':>5s} {'vs RS':>7s}  {'Status':<14s}")
print('-' * 70)
for label, n, target, peak, peak_ep, ratio, status in rows:
    plan = str(target)
    print(f"{label:<14s} {n:>6d} {plan:>6s} {peak:>7.1f}N {str(peak_ep):>5s} {ratio:>6.2f}x  {status:<14s}")


## 7.3 Best-mask gallery

Visual snapshot of the highest-reward mask each method produced. Helps surface qualitative differences in the topologies the policies converged on.


In [ ]:
# One panel per run: the best-reward mask
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RUNS = [
    ('DQN',            'designs/dqn_run'),
    ('PPO-v2',         'designs/ppo_v2_run'),
    ('A2C-v2',         'designs/a2c_v2_run'),
    ('SB3-PPO',        'designs/sb3_ppo_run'),
    ('SB3-A2C',        'designs/sb3_a2c_run'),
    ('PPO (raw)',      'designs/ppo_run'),
    ('A2C (raw)',      'designs/a2c_run'),
]

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
axes = axes.flatten()

for ax, (label, d) in zip(axes, RUNS):
    s = json.loads(open(f'{d}/training_stats.json').read())
    eps = s['episode_stats']; hp = s['hyperparams']
    f = np.array([e['force_N'] for e in eps])
    best_ep = int(np.argmax(f)) + 1   # use peak-force as 'best' for visual fairness
    best_F  = f.max(); best_iron = eps[best_ep - 1]['iron_count']
    img_path = Path(d) / 'episodes' / f'ep_{best_ep:03d}_mask.png'
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path)); ax.axis('off')
        ax.set_title(f'{label}\nep {best_ep}: {best_F:.0f} N, {best_iron} cells', fontsize=10)
    else:
        ax.text(0.5, 0.5, 'mask not found', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        ax.set_title(label, fontsize=10)

# Hide unused axes
for ax in axes[len(RUNS):]:
    ax.axis('off')

plt.suptitle('Best-mask gallery: peak-force topology per method', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 7.4 Why DQN beat the policy methods

DQN's peak (1305 N) is roughly **1.6×** what PPO-v2 achieved (795 N) and **1.7×** A2C-v2's plateau (764 N), on the same env, same backbone, same reward function. The reason is grounded in the action-effect probe from Section 6.8 — *not* in algorithmic sophistication.

**Recall the action-effect finding.** Single-step rewards do not distinguish the 4 controller actions: all four actions, taken as the first step from random starts, produce overlapping force distributions with means within 3 N of each other (σ ≈ 10 N). The useful signal lives entirely in *multi-step compositions* — sequences of 34 actions that happen to assemble into productive topologies.

**How each algorithm extracts signal from this landscape:**

| Algorithm | Gradient updates per FEMM solve | Mechanism |
|---|---:|---|
| A2C | ~1/34 (one update per 34-step episode) | One MC return, no replay |
| PPO | ~4/34 (K=4 epochs on the same rollout) | Limited rollout reuse via clipped surrogate |
| **DQN** | **~30 per transition on average** | Replay buffer: each FEMM result is drawn into ~30 different minibatches before being evicted |

DQN's replay buffer is the dominant factor. Every FEMM solve produces one $(s, a, r, s')$ tuple that gets revisited dozens of times by the Q-learning updates, with the n-step bootstrapped TD target slowly propagating the eventual force back through 34 steps of state evolution. That is exactly the kind of mechanism needed when the per-step reward is uninformative — the credit-assignment problem is *delegated to the bellman target*, not to the policy gradient.

PPO is bounded by its on-policy assumption: once the policy moves, the old rollouts are stale and must be discarded. A2C is the extreme case: 1 gradient step per FEMM solve. The action-effect landscape punishes thin signal-reuse, and this shows up cleanly in the 200-episode windows: PPO-v2 entropy decays 1.35 → 1.19 → 0.86 → 0.84 with mean force climbing 132 → 147 → 200 → 231 N — slow but real on-policy progress, not collapse. A2C-v2 entropy stays at 1.27–1.37 (≈ uniform over 4 actions; log 4 ≈ 1.39) with no mean-force trend across all four windows — the single MC-return-per-episode gradient is being washed out by reward variance and the policy never commits. DQN, meanwhile, keeps making progress through ep 200; the running-best curves in Section 7.1 show the gap directly.

**The reward-scaling fix matters for the policy methods, not for DQN.** DQN uses a smooth-L1 loss on a TD target rather than a value-MSE-vs-MC-return loss, so the $10^7$-scale value-loss explosion that derails A2C/PPO simply does not happen. The unscaled runs of A2C and PPO collapsed; their scaled `_v2` counterparts trained productively. DQN trained productively in both regimes because the loss structure is fundamentally different.


## 7.5 Computational cost

Wall-clock and FEMM-solve totals for every run. FEMM solve time was measured empirically at ~84 s/solve (random-search probe: 200 solves in 16,752 s).


In [ ]:
# Compute approximate wall-clock and FEMM-solve counts per run
import json
from pathlib import Path

SEC_PER_SOLVE = 84.0   # empirical from random-search probe
STEPS_PER_EP = 34

RUNS = [
    ('DQN',            'designs/dqn_run'),
    ('PPO-v2',         'designs/ppo_v2_run'),
    ('A2C-v2',         'designs/a2c_v2_run'),
    ('SB3-PPO',        'designs/sb3_ppo_run'),
    ('SB3-A2C',        'designs/sb3_a2c_run'),
    ('PPO (raw)',      'designs/ppo_run'),
    ('A2C (raw)',      'designs/a2c_run'),
    ('random-search',  'designs/env_probe/random_search'),
    ('action-effect',  'designs/env_probe/action_effect'),
]

print(f"{'Run':<16s} {'eps':>5s} {'FEMM solves':>13s} {'wall (h)':>10s}")
print('-' * 50)
for label, d in RUNS:
    if d.endswith('random_search'):
        files = sorted(Path('designs/env_probe').glob('random_search_*.json'))
        if files:
            s = json.loads(files[-1].read_text()); solves = s['summary']['n']
            print(f"{label:<16s} {'-':>5s} {solves:>13d} {solves*SEC_PER_SOLVE/3600:>10.1f}")
    elif d.endswith('action_effect'):
        files = sorted(Path('designs/env_probe').glob('action_effect_*.json'))
        if files:
            s = json.loads(files[-1].read_text()); solves = s['reps'] * 4
            print(f"{label:<16s} {'-':>5s} {solves:>13d} {solves*SEC_PER_SOLVE/3600:>10.1f}")
    else:
        s = json.loads(open(f'{d}/training_stats.json').read())
        n = len(s['episode_stats']); solves = n * STEPS_PER_EP
        print(f"{label:<16s} {n:>5d} {solves:>13d} {solves*SEC_PER_SOLVE/3600:>10.1f}")

print()
print(f'(Reference: empirical FEMM solve cost = {SEC_PER_SOLVE:.0f} s/solve on the Windows training box.)')


**Interpretation.** The DQN run consumed ~159 h of FEMM wall-clock for 6,800 solves. Random-search produced its 610 N ceiling in ~4.7 h with 200 solves — i.e. **for the price of one DQN run you could run ~30 random-search probes**. The justification for DQN's compute cost is the $2.14\times$ improvement over random-search peak; whether that ratio is worth $\sim 30\times$ compute is a problem-specific judgement that depends on whether the user needs *the* best topology or merely *a* good one.

Critically, DQN is the only run in the table whose wall-clock fully amortizes against its result — the policy-method runs that collapsed (A2C raw, PPO raw) spent compute producing data that's only useful as negative controls; SB3-A2C burned 79 h of FEMM time to be stuck at an episode-2 best.


## 7.6 Hyperparameter table

Configuration of every run, pulled from the `hyperparams` block of each `training_stats.json`. Useful for reviewers and for any future reproduction attempt.


In [ ]:
# Print the hyperparameter table
import json

RUNS = [
    ('DQN',           'designs/dqn_run'),
    ('A2C (raw)',     'designs/a2c_run'),
    ('A2C-v2',        'designs/a2c_v2_run'),
    ('PPO (raw)',     'designs/ppo_run'),
    ('PPO-v2',        'designs/ppo_v2_run'),
    ('SB3-A2C',       'designs/sb3_a2c_run'),
    ('SB3-PPO',       'designs/sb3_ppo_run'),
]

fields = [
    'alpha', 'gamma', 'gae_lambda', 'clip_eps', 'ppo_epochs',
    'epsilon_start', 'epsilon_min', 'epsilon_decay', 'target_tau',
    'batch_size', 'replay_size', 'entropy_coef', 'entropy_coef_start',
    'entropy_coef_final', 'value_coef', 'reward_scale', 'reward_kind',
    'max_steps', 'timesteps',
]

table = {}
for label, d in RUNS:
    hp = json.loads(open(f'{d}/training_stats.json').read())['hyperparams']
    table[label] = {f: hp.get(f, '-') for f in fields}

# Print as a wide column-major table; only show fields that are non-trivial somewhere
shown = [f for f in fields if any(table[k][f] != '-' for k in table)]

head = f"{'Hyperparameter':<22s}" + ''.join(f"{k:>11s}" for k in table)
print(head)
print('-' * len(head))
for f in shown:
    row = f"{f:<22s}"
    for k in table:
        v = table[k][f]
        if isinstance(v, float):
            cell = f'{v:g}'
        else:
            cell = str(v)
        row += f"{cell:>11s}"
    print(row)


## 7.7 What's missing for a defensible claim

To say "DQN beats every alternative on this problem" with full rigor, the following runs would be needed — flagged here as planned future work:

1. **DQN evaluation pass.** Load `designs/dqn_run/checkpoint.pt`, fix the policy (greedy argmax), and roll out 20-50 evaluation episodes from a held-out set of random starts. The mean force from that pass is the number that should populate any benchmark — the 1305 N peak is an *upper bound the agent has demonstrated reachable*, not a steady-state operating point.
2. **PPO-v2 continuation past 200 eps.** The 200-episode run shown above ended with entropy still dropping (0.84) and mean force still climbing across the four 50-ep windows (132 → 147 → 200 → 231 N), peak 795 N. The trajectory is not settled, but is also not on a path to overtake DQN within another 100 eps. A 400- or 500-episode continuation with `--clip-eps 0.1 --entropy-coef-final 0.02` is the one remaining experiment that could flip the "policy methods cannot match DQN on this env" claim; it is also the only such experiment whose data justifies the spend.
3. **SB3-A2C with reward scaling.** SB3-A2C currently underperforms because it was run with `reward_scale=1.0`. A fair head-to-head requires SB3-A2C with `--reward-scale 0.001`.
4. **Random-policy floor.** The `env_probe.py random-policy` subcommand crashed at ep 13/50 — its result is needed to bracket the action-effect interpretation from below.
5. **Cell-flips landscape.** `env_probe.py cell-flips --from-best designs/dqn_run/best_mask.npy` would characterize the local reward surface around the DQN best — useful to argue *what* a productive topology looks like rather than just *how much force* it produces.

Items 1, 2, 3 are wall-clock-bounded only; items 4, 5 require an env-probe rerun.


## 7.8 Summary

What the data on disk supports:

- **DQN achieved 1305 N peak end-of-episode force**, 2.14× the random-search ceiling (610 N) and 33× the 40 N conventional C-core baseline (Chapter 4 spec). This is the strongest result in the chapter.
- **Reward scaling is a necessary intervention for stable A2C / PPO on this env.** Without it, both algorithms collapse in $\sim 50$ episodes (entropy → $10^{-20}$, value-loss → $10^7$). With it, PPO-v2 trains to $\sim 795$ N and A2C-v2 plateaus near $\sim 764$ N. The scaling story is documented in Section 5.9.
- **DQN's advantage over the policy methods is grounded in the env's action-effect structure** (Section 6.8): single-step rewards do not distinguish actions, so credit assignment must come from multi-step signal compounding. DQN's replay buffer revisits each FEMM solve dozens of times across n-step TD updates; on-policy A2C / PPO cannot do this and accordingly plateau or undertrain.
- **The SB3 reference implementations underperform our custom scripts under default config.** This is consistent with the reward-scaling failure mode: SB3-A2C with `reward_scale=1.0` is functionally equivalent to our `a2c_run`. A fair SB3 comparison would require rerunning with `--reward-scale 0.001`; that rerun is tracked as planned work in Section 7.7.

What the data on disk does NOT yet support:

- Steady-state policy performance numbers for any method (no evaluation passes — all numbers are training-time bests).
- A claim that DQN's $2.14\times$ over random search is the *best possible* — only one DQN seed was run, with no hyperparameter sweep.
- A clean DRL-vs-GA comparison on Newton-calibrated forces — Chapter 4's GA numbers are legacy thesis figures, not recomputed here.
- A reason to spend more compute on A2C-v2 — its 200-episode trace shows entropy stuck at 1.27–1.37 (near uniform over 4 actions) and no mean-force trend across the run. The plateau is a property of the algorithm on this env, not of the training budget; additional A2C-v2 episodes are *not* recommended.


## References

- Mnih, V. et al. *Human-level control through deep reinforcement learning* (Nature, 2015) — DQN reference, including replay buffer + target network mechanics that this chapter relies on.
- Schulman, J. et al. *Proximal Policy Optimization Algorithms* (arXiv:1707.06347, 2017) — PPO reference, including the clipped surrogate and approximate-KL diagnostic that Section 5.9 leans on.
- Stable-Baselines3: [https://stable-baselines3.readthedocs.io/](https://stable-baselines3.readthedocs.io/) — reference implementations used for the SB3-PPO and SB3-A2C runs.
